# Part 2 — Retrieval-Augmented Generation (LangChain, 10 movie Wikipedia pages)

**Rubric map**
1. Load 10 docs with a LangChain loader &rarr; *Section 2*
2. Split (chunk_size 500, overlap 50) &rarr; *Section 3*
3. Embed chunks &rarr; Vector Store &rarr; *Section 4*
4. Explicit pipeline: PromptTemplate + retriever + LLM (no pre-built chain) &rarr; *Section 5*
5. 5 questions through retriever + LLM; show retrieved chunks + answer &rarr; *Section 6*
6. Change chunk size/overlap for 2 questions, re-run, compare &rarr; *Section 7*
7. Retrieval eval: top-3 contains answer? (Y/N) + rank; Retrieval Success Rate &rarr; *Section 8*
8. Analyse &ge;2 RAG failures &rarr; *Section 9*

> **Run in Colab.** Uses a local `flan-t5` LLM (no API key) + a local
> sentence-transformer embedder, so nothing paid is required.


## 1. Install & imports

In [2]:
!pip install -q langchain langchain-community langchain-huggingface \
    sentence-transformers faiss-cpu wikipedia transformers


In [3]:
import pandas as pd
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline


C:\Users\Rohil\AppData\Local\Temp\ipykernel_34460\4030129736.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


## 2. Load 10 movie Wikipedia pages

A couple of titles are deliberately disambiguated (`Titanic (1997 film)`,
`Parasite (2019 film)`, `Gladiator (2000 film)`) so the loader grabs the film,
not the ship / organism / Roman fighter. This ambiguity is revisited in the
failure analysis.

In [7]:
MOVIES = [
    "The Matrix", "Inception", "The Godfather", "Pulp Fiction",
    "Titanic (1997 film)", "The Dark Knight (film)", "Forrest Gump",
    "Interstellar (film)", "Parasite (2019 film)", "Gladiator (2000 film)",
]

docs = []
for title in MOVIES:
    d = WikipediaLoader(query=title, load_max_docs=1, doc_content_chars_max=20000).load()
    import wikipedia
    wikipedia.set_rate_limiting(True)   # spaces out requests automatically
    if d:
        d[0].metadata["movie"] = title
        docs.append(d[0])
    print(f"{title:28s} -> {'OK' if d else 'MISSING'}  "
          f"({len(d[0].page_content) if d else 0} chars)")

print("\nLoaded", len(docs), "documents")

The Matrix                   -> OK  (20000 chars)
Inception                    -> OK  (20000 chars)
The Godfather                -> OK  (20000 chars)
Pulp Fiction                 -> OK  (20000 chars)
Titanic (1997 film)          -> OK  (20000 chars)
The Dark Knight (film)       -> OK  (20000 chars)
Forrest Gump                 -> OK  (20000 chars)
Interstellar (film)          -> OK  (20000 chars)
Parasite (2019 film)         -> OK  (20000 chars)
Gladiator (2000 film)        -> OK  (20000 chars)

Loaded 10 documents


## 3. Split into chunks (size 500 / overlap 50)

In [8]:
def make_chunks(documents, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""])
    return splitter.split_documents(documents)

chunks = make_chunks(docs, 500, 50)
print("Total chunks:", len(chunks))
print("Example chunk:\n", chunks[0].page_content[:300], "...")


Total chunks: 605
Example chunk:
 The Matrix is a 1999 science fiction action film written and directed by the Wachowskis. The first installment in the Matrix film series, it stars Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss, Hugo Weaving, and Joe Pantoliano. It depicts a dystopian future in which humanity is unknowingly trap ...


## 4. Embed chunks &rarr; FAISS vector store

In [9]:
import torch
_emb_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding on:", _emb_device)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": _emb_device},
)

def build_store(chunk_list):
    return FAISS.from_documents(chunk_list, embeddings)

vectorstore = build_store(chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})   # top-3
print("Vector store ready with", vectorstore.index.ntotal, "vectors")


Embedding on: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready with 605 vectors


## 5. Explicit pipeline: PromptTemplate + retriever + LLM

We wire the pieces together **by hand** (retrieve &rarr; format context &rarr; fill
prompt &rarr; call LLM) instead of using a prebuilt `RetrievalQA` chain.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate

_model_name = "google/flan-t5-base"
_tok = AutoTokenizer.from_pretrained(_model_name)
_model = AutoModelForSeq2SeqLM.from_pretrained(_model_name)
_device = "cuda" if torch.cuda.is_available() else "cpu"
_model = _model.to(_device)
print("LLM on:", _device)

def llm_generate(prompt_text, max_new_tokens=80):
    inputs = _tok(prompt_text, return_tensors="pt",
                  truncation=True, max_length=1024).to(_device)
    with torch.no_grad():
        out = _model.generate(**inputs, max_new_tokens=max_new_tokens)
    return _tok.decode(out[0], skip_special_tokens=True).strip()

PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Use ONLY the context below to answer the question. "
        "If the answer is not in the context, say 'I don't know'.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\nAnswer:"
    ),
)

def rag_answer(question, retriever, prompt=PROMPT):
    retrieved = retriever.invoke(question)
    context = "\n\n".join(
        f"[Chunk {i+1} | {d.metadata.get('movie','?')}] {d.page_content}"
        for i, d in enumerate(retrieved))
    final_prompt = prompt.format(context=context, question=question)
    answer = llm_generate(final_prompt)
    return answer, retrieved

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM on: cuda


## 6. Five questions &rarr; retrieved chunks + generated answer

In [11]:
QUESTIONS = [
    "Who directed Inception?",
    "In what year was the movie Titanic released?",
    "Who played the Joker in The Dark Knight?",
    "Who directed the movie Parasite?",
    "Who composed the score for Interstellar?",
]
EXPECTED = ["Nolan", "1997", "Ledger", "Bong Joon", "Zimmer"]   # "Bong Joon" matches the spaced spelling
def show(question, retriever):
    ans, docs = rag_answer(question, retriever)
    print("Q:", question)
    print("A:", ans)
    for i, d in enumerate(docs, 1):
        snippet = d.page_content.replace("\n", " ")[:180]
        print(f"   chunk {i} [{d.metadata.get('movie','?')}]: {snippet}...")
    print("-" * 90)
    return ans, docs

results = {q: show(q, retriever) for q in QUESTIONS}

Q: Who directed Inception?
A: Christopher Nolan
   chunk 1 [Inception]: Inception is a 2010  science fiction film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professio...
   chunk 2 [Inception]: . Nolan took a long time re-writing the script in order "to make sure that the emotional journey of his [DiCaprio's] character was the driving force of the movie." On February 11, ...
   chunk 3 [Inception]: . in February 2009. Inception was filmed in six countries, beginning in Tokyo on June 19 and ending in Canada on November 22. Its official budget was US$160 million, split between ...
------------------------------------------------------------------------------------------
Q: In what year was the movie Titanic released?
A: 1997
   chunk 1 [Titanic (1997 film)]: Titanic is a 1997 American epic historical romance film written and directed by James Cameron. Incorporating both historical and fictional aspects,

## 7. Change chunk size/overlap for 2 questions &rarr; compare

We rebuild the store with **larger** chunks (size 1000 / overlap 200) and re-run
questions 2 and 5, then compare retrieved chunks and answers to the original
500/50 configuration.

In [12]:
chunks_big = make_chunks(docs, 1000, 200)
store_big = build_store(chunks_big)
retriever_big = store_big.as_retriever(search_kwargs={"k": 3})
print("Big-chunk store:", store_big.index.ntotal, "vectors (vs", vectorstore.index.ntotal, "original)")

compare_qs = [QUESTIONS[1], QUESTIONS[4]]   # Titanic year, Interstellar score
for q in compare_qs:
    print("\n================", q, "================")
    print(">>> ORIGINAL (500/50)")
    a0, d0 = show(q, retriever)
    print(">>> LARGER (1000/200)")
    a1, d1 = show(q, retriever_big)
    print(f"answer changed? {a0.strip() != a1.strip()}   "
          f"(orig='{a0.strip()}'  large='{a1.strip()}')")

Big-chunk store: 329 vectors (vs 605 original)

================ In what year was the movie Titanic released? ================
>>> ORIGINAL (500/50)
Q: In what year was the movie Titanic released?
A: 1997
   chunk 1 [Titanic (1997 film)]: Titanic is a 1997 American epic historical romance film written and directed by James Cameron. Incorporating both historical and fictional aspects, it is based on accounts of the s...
   chunk 2 [Titanic (1997 film)]: Titanic premiered at the Tokyo International Film Festival on November 1, 1997, and was released in the United States on December 19. It was distributed by Paramount Pictures in th...
   chunk 3 [Titanic (1997 film)]: . Scale models, computer-generated imagery (CGI), and a reconstruction of the Titanic built at Baja Studios were used to recreate the sinking. Titanic was initially in development ...
------------------------------------------------------------------------------------------
>>> LARGER (1000/200)
Q: In what year was the movi

In [13]:
# Adversarial questions to surface genuine RAG failure modes (rubric Section 9)
FAILURE_QS = [
    "Who wrote the score?",                          # ambiguous: no movie named
    "Which movie features a spinning top as a totem?",  # oblique: answer not stated in those words
    "How much money did the heist crew steal?",      # cross-movie / not in docs
]
for q in FAILURE_QS:
    show(q, retriever)

Q: Who wrote the score?
A: Nino Rota
   chunk 1 [The Godfather]: . The score was composed principally by Nino Rota, with additional pieces by Carmine Coppola....
   chunk 2 [Parasite (2019 film)]: The score, by South Korean composer Jung Jae-il, consisted of "minimalist piano pieces, punctuated with light percussion", setting the film's "tense atmosphere". It also had a baro...
   chunk 3 [Gladiator (2000 film)]: . In addition to Crowe's acting and Scott's directing, some reviewers also applauded John Mathieson's cinematography, Arthur Max's production design, and the musical score composed...
------------------------------------------------------------------------------------------
Q: Which movie features a spinning top as a totem?
A: Parasite
   chunk 1 [Inception]: Joseph Gordon-Levitt, who plays Arthur, spent several weeks learning to fight in a corridor that spun like "a giant hamster wheel". Nolan said of the device, "It was like some incr...
   chunk 2 [Inception]: . Cobb uses a

**Result (this run).** Both questions were unchanged by the larger chunks:

- **Titanic year** — answer stayed **1997** at both 500/50 and 1000/200; the same
  premiere/release chunk was retrieved.
- **Interstellar composer** — answer stayed **Hans Zimmer**; at 1000/200 the
  relevant chunk simply absorbed its `=== Music ===` section header, i.e. more
  surrounding context per chunk, but the ranking and answer did not change.

**Takeaway:** for these unambiguous, single-fact questions retrieval was *robust*
to chunk size. Larger chunks pull in more context per hit (which can help facts
that live a sentence away from the entity) but also dilute the embedding and can
rank a broad chunk above a precise one — neither effect flipped the answer here.


## 8. Retrieval evaluation & Success Rate

For each question we check whether **any of the top-3 chunks** contains the
correct answer, and the **rank** of the first relevant chunk. The automated
check uses an answer keyword as a proxy for the "manual" relevance judgement
&mdash; **verify each row by eye** before reporting.

> **Manual-verification note.** The keyword check is literal substring matching,
> so a spelling mismatch can produce a *false* "No". Wikipedia writes the Parasite
> director as "Bong Joon Ho" (space), not "Bong Joon-ho" (hyphen), so the hyphenated
> keyword initially reported a spurious miss. Inspecting the retrieved chunks by
> hand confirms the correct Parasite chunk was retrieved at rank 1, so the keyword
> is set to `"Bong Joon"` and the true **Retrieval Success Rate = 5/5 = 100%**.


In [14]:
rows = []
for q, exp in zip(QUESTIONS, EXPECTED):
    _, docs = results[q]
    first_rank, hit = None, "No"
    for rank, d in enumerate(docs, 1):
        if exp.lower() in d.page_content.lower():
            hit, first_rank = "Yes", rank
            break
    rows.append({"question": q, "answer_keyword": exp,
                 "top3_contains": hit, "first_relevant_rank": first_rank})

eval_tbl = pd.DataFrame(rows)
display(eval_tbl)

success = (eval_tbl["top3_contains"] == "Yes").sum()
print(f"Retrieval Success Rate = {success}/5 = {success/5:.0%}")


,question,answer_keyword,top3_contains,first_relevant_rank
0,Who directed Inception?,Nolan,Yes,1
1,In what year was the movie Titanic released?,1997,Yes,1
2,Who played the Joker in The Dark Knight?,Ledger,Yes,1
3,Who directed the movie Parasite?,Bong Joon,Yes,1
4,Who composed the score for Interstellar?,Zimmer,Yes,2


Retrieval Success Rate = 5/5 = 100%


## 9. RAG failure analysis (&ge;2)

Run the cell below to auto-surface candidate failures, then write up **at least
two**. Common failure modes to look for:

- **Correct document not retrieved** — the right movie page never appears in top-3.
- **Relevant chunk ranked too low** — answer is in chunk 3, not chunk 1.
- **Irrelevant chunk ranked highly** — a plot-summary chunk outranks the fact chunk.
- **Correct context, wrong answer** — flan-t5 reads the right chunk but still answers wrong.
- **Unsupported answer** — the LLM adds a fact not in the context (hallucination).
- **Chunk boundary split** — e.g. "score composed by **Hans / Zimmer**" gets cut
  across two chunks so neither chunk is fully relevant.
- **Ambiguous name** — "score" (music vs rating vs sports), or an under-disambiguated
  film title retrieving the wrong entity.

In [15]:
print("Candidate failures (verify manually):\n")
for _, r in eval_tbl.iterrows():
    ans = results[r["question"]][0]
    problem = []
    if r["top3_contains"] == "No":
        problem.append("relevant chunk NOT in top-3")
    elif r["first_relevant_rank"] and r["first_relevant_rank"] > 1:
        problem.append(f"relevant chunk ranked #{r['first_relevant_rank']}")
    if r["answer_keyword"].lower() not in ans.lower():
        problem.append(f"LLM answer '{ans}' missing expected keyword")
    if problem:
        print("-", r["question"])
        print("   ->", "; ".join(problem), "\n")


Candidate failures (verify manually):

- Who composed the score for Interstellar?
   -> relevant chunk ranked #2 



### Failure analysis — two genuine failures (from the adversarial questions)

The five rubric questions all succeeded (5/5), so the auto-surfaced list above only
notes a *minor* retrieval observation — the Interstellar composer sat at rank&nbsp;2
rather than rank&nbsp;1 (still a success, correct answer). The two genuine failures
below come from the deliberately harder adversarial questions.

**Failure A — Ambiguous query, no entity to disambiguate: "Who wrote the score?"**
With no film named, the retriever has nothing to anchor on and returns *score*
chunks from three different movies — The Godfather (Nino Rota), Parasite
(Jung Jae-il), and Gladiator. The LLM answered **"Nino Rota"**: correct for *one*
film, but arbitrary — it reflects whichever music chunk embedded closest, not the
user's intent. *Failure mode: ambiguous / under-specified query → arbitrary
document selection.* *Mitigation:* require the movie name in the query, or add a
title/section metadata filter so the retriever scopes to one film.

**Failure B — Correct context retrieved, LLM answered wrong: "Which movie
features a spinning top as a totem?"** The correct answer is **Inception**, and
retrieval *succeeded*: the top two chunks were both Inception, with chunk&nbsp;2
explicitly describing Cobb's spinning top that "spins indefinitely in a dream."
Yet the model answered **"Parasite"** — a film that appeared only at rank&nbsp;3
and is unrelated to the totem. *Failure mode: generation error despite correct
retrieval* — the small `flan-t5-base` model mis-attributed the fact across a
mixed multi-movie context. This is an **LLM** failure, not a retrieval failure.
*Mitigation:* a stronger model (`flan-t5-large` or an API LLM), per-movie
retrieval so the context isn't mixed, or a more constrained prompt.

**Counter-example (grounding worked): "How much money did the heist crew steal?"**
No document states a heist theft amount, and the model correctly answered
**"I don't know"** rather than inventing a figure — showing the "if not in the
context, say I don't know" instruction suppressed hallucination.
